# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es establecer un baseline con Multinomial Bayes Naive sobre dos representaciones de texto, Bag of Words y TF-IDF, con búsqueda de hiperparámetros.

NB es el modelo canónico para clasificación de texctos, es rápido de entrenar y transparente, ya que los pesos son log-probabilidades por palabra y clase.

Cualquier modelo mas sofisticado debe superar el F1-macro de NB, si no lo hace hay un bug.

## Importación e instalación de dependencias


In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N5: Bayes Naive + BoW con GridSearch

Las entregas anteriores mostraron que BoW con priors uniformes (N4)
supera levemente a TF-IDF con GridSearch (N3): F1-macro 0.6533 vs 0.6525.
Esta entrega combina ambas ideas: aplica búsqueda exhaustiva de
hiperparámetros sobre el pipeline BoW + MultinomialNB.

Se exploran los siguientes hiperparámetros:

- **`bow__ngram_range`**: unigramas solos `(1,1)` vs unigramas + bigramas
  `(1,2)`. Los bigramas capturan expresiones como "no funciona" o "muy bueno"
  que son centrales para el análisis de sentimiento.

- **`bow__max_features`**: tamaño del vocabulario (30.000 o 50.000 tokens).
  Un vocabulario más chico actúa como regularización implícita al descartar
  términos raros o ruidosos.

- **`nb__alpha`**: suavizado de Laplace. Valores bajos (0.1) confían más en
  los conteos del corpus; valores altos (2.0) suavizan más hacia una
  distribución uniforme, lo que ayuda cuando hay palabras poco frecuentes.

- **`nb__fit_prior`**: si el modelo aprende los priors de clase del dataset
  (`True`, resultando en ~40% neg, ~20% neutra, ~40% pos) o los fuerza
  uniformes (`False`, 1/3 cada clase). La N4 mostró que los priors uniformes
  mejoran la detección de la clase neutra, que está subrepresentada.

La búsqueda se realiza con validación cruzada de 5 folds optimizando F1-macro,
que es la métrica de la competencia Kaggle.

In [ ]:
pipe = Pipeline([
    ("bow", make_bow()),
    ("nb",  MultinomialNB())
])

param_grid_bow_v1 = {
    "bow__ngram_range": [(1, 1), (1, 2)], # Rango de ngramas
    "bow__min_df": [3, 5, 10], # Cantidad minima de documentos donde debe aparecer la palabra
    "bow__max_features": [30_000, 50_000, 100_000], # Limite del vocabulario
    "nb__alpha": [0.1, 0.2, 0.3, 0.5], # Suavizado de Laplace
    "nb__fit_prior": [True, False]  # False = priors uniformes
}

gs_bow_v1 = GridSearchCV(
    pipe,
    param_grid_bow_v1,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1
)

print("Buscando mejores hiperparámetros v1...")
gs_bow_v1.fit(X_train, y_train)

print("Mejores hiperparámetros:", gs_bow_v1.best_params_)
print(f"Mejor F1-macro en CV: {gs_bow_v1.best_score_:.4f}")

Buscando mejores hiperparámetros v1...
Fitting 5 folds for each of 144 candidates, totalling 720 fits
Mejores hiperparámetros: {'bow__max_features': 50000, 'bow__min_df': 3, 'bow__ngram_range': (1, 2), 'nb__alpha': 0.5, 'nb__fit_prior': False}
Mejor F1-macro en CV: 0.6493


In [ ]:
y_pred = gs_bow_v1.best_estimator_.predict(X_val)
evaluate("nb_bow_gridsearch_power_v1", y_val, y_pred, hyperparams=gs_bow_v1.best_params_)


=== nb_bow_gridsearch_power_v1 ===
Hiperparámetros: {'bow__max_features': 50000, 'bow__min_df': 3, 'bow__ngram_range': (1, 2), 'nb__alpha': 0.5, 'nb__fit_prior': False}

F1-macro:  0.7915
Precision: 0.7885
Recall:    0.8022
Accuracy:  0.8119

              precision    recall  f1-score   support

    negativa     0.8732    0.8032    0.8367      4080
      neutra     0.5943    0.7539    0.6646      2040
    positiva     0.8982    0.8495    0.8732      4080

    accuracy                         0.8119     10200
   macro avg     0.7885    0.8022    0.7915     10200
weighted avg     0.8274    0.8119    0.8169     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3277     647       156
neutra         265    1538       237
positiva       211     403      3466


In [ ]:
best_pipe = gs_bow_v1.best_estimator_
best_pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(best_pipe, "nb_bow_gridsearch_power_v1")
make_submission(best_pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_nb_bow_gridsearch_power_v1.csv")

Modelo guardado → models/nb_bow_gridsearch_power_v1.joblib
Predicción guardada → submissions/submission_nb_bow_gridsearch_power_v1.csv  (8500 predicciones)
Distribución: clase 0: 37.1%, clase 1: 25.3%, clase 2: 37.6%


PosixPath('submissions/submission_nb_bow_gridsearch_power_v1.csv')